In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/chiaracervelli/tweets/tweets.csv
/kaggle/input/datasets/chiaracervelli/embedding/base_map.csv
/kaggle/input/datasets/chiaracervelli/embedding/umap_mapper.joblib
/kaggle/input/datasets/chiaracervelli/embedding/embeddings_1024d.pt
/kaggle/input/datasets/chiaracervelli/embedding/topics_bertopic.json


# Personal setting

In [2]:
BASE_INPUT_PATH = "/kaggle/input/datasets/chiaracervelli/"
EMBEDDING_DIR = "embedding" # Se hai creato una cartella apposita
TWEETS_DIR = "tweets"
BASE_WORKING_PATH = "/kaggle/working/"
PORT= 5500

## Install dependencies 

In [3]:
!pip install umap-learn spacy nltk
!python -m spacy download en_core_web_sm
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.2 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 79.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled hugg

In [4]:
!pip install flask flask-cors pyngrok


In [5]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.5 MB/s eta 0:00:00


## Imports

In [6]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
import umap
import spacy
from nltk.corpus import wordnet
import nltk
from tqdm import tqdm
import os
import joblib 
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
import threading
import json
import requests

# Download WordNet if not present
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

2026-05-16 12:52:32.912204: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778935953.123264     104 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778935953.182627     104 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778935953.665988     104 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778935953.666036     104 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778935953.666039     104 computation_placer.cc:177] computation placer alr

Using device: cuda



In [7]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient
# Tell pyngrok to use the cleanly installed apt version
conf.get_default().ngrok_path = "/usr/bin/ngrok"

# Safely retrieve the token from your Kaggle Secrets vault
user_secrets = UserSecretsClient()
my_secret_token = user_secrets.get_secret("NGROK_TOKEN")

In [8]:
global_vectorizer = None
global_feature_names = None

## Model initialization

In [9]:
# 1. Load Embedding Model (Qwen 0.6B)
emb_model_id = "Qwen/Qwen3-Embedding-0.6B"
print(f"Loading embedding model {emb_model_id}...")
emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_id, trust_remote_code=True)
emb_tokenizer.padding_side = "right"
emb_model = AutoModel.from_pretrained(emb_model_id, trust_remote_code=True).to(device)
emb_model.eval()

# 2. Load Generative Model (Gemma 4 E2B Instruct)
gen_model_id = "google/gemma-4-E2B-it"
print(f"Loading generative model {gen_model_id}...")

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_id)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_id, 
    torch_dtype=torch.bfloat16,
    device_map="cuda:0"
)



Loading embedding model Qwen/Qwen3-Embedding-0.6B...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

... (7 outputs left out)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

## Embedding functions

In [10]:
def get_embedding(text):
    """Calculates the embedding for a single text string."""
    inputs = emb_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = emb_model(**inputs)
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embedding = sum_embeddings / sum_mask
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding

def batch_get_embeddings(texts, batch_size=32):
    """Calculates embeddings for a list of texts efficiently in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Calculating Batch Embeddings"):
        batch_texts = texts[i : i + batch_size]
        inputs = emb_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = emb_model(**inputs)
            attention_mask = inputs['attention_mask']
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embeddings = sum_embeddings / sum_mask
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

## Perturbation logic

In [11]:

def get_wordnet_pos_from_spacy(spacy_pos):
    """Maps spaCy POS tags to WordNet formats."""
    if spacy_pos == 'ADJ': return wordnet.ADJ
    elif spacy_pos == 'VERB': return wordnet.VERB
    elif spacy_pos == 'NOUN': return wordnet.NOUN
    elif spacy_pos == 'ADV': return wordnet.ADV
    return None
def perturb_dict(text, mode="synonym", target_word=None, replacement_word=None):
    doc = nlp(text)
    perturbed_text = ""
    display_text = "" 
    changed = False
    target_found = False

    if mode == "custom":
        if not target_word:
            raise ValueError("For the custom replace you should select a word in the tweet first.")
        if not replacement_word or not replacement_word.strip():
            raise ValueError("Write the new word.")

    for token in doc:
        # 1. SCENARIO: The user HAS CHOSEN a target word
        if target_word:
            if token.text.lower() == target_word.lower():
                target_found = True

                if mode == "custom":
                    perturbed_text += replacement_word + token.whitespace_
                    display_text += replacement_word + token.whitespace_
                    changed = True
               
                elif mode == "remove":
                    perturbed_text += token.whitespace_
                    display_text += token.whitespace_
                    changed = True
                else:
                     # Find the synonym or antonym
                    replacement = get_single_word_replacement(token, mode=mode)
                    if replacement:
                        perturbed_text += replacement + token.whitespace_
                        display_text += f"<strong>{replacement}</strong>{token.whitespace_}"
                        changed = True
                    else:
                        # Target found, but WordNet has no replacement for it
                        perturbed_text += token.text_with_ws
                        display_text += token.text_with_ws
            else:
                # It's a different word, copy it and move on
                perturbed_text += token.text_with_ws
                display_text += token.text_with_ws

        # 2. SCENARIO: The user HAS NOT CHOSEN anything (Automatic Mode)
        else:
            # Here we keep the rule: change only the FIRST valid word you find, then stop
            if changed or not token.is_alpha:
                perturbed_text += token.text_with_ws
                display_text += token.text_with_ws
                continue
            if mode == "remove":
                perturbed_text += token.whitespace_
                display_text += token.whitespace_
                changed = True
            else:
                replacement = get_single_word_replacement(token, mode=mode)
                if replacement:
                    perturbed_text += replacement + token.whitespace_
                    display_text += replacement + token.whitespace_
                    changed = True
                else:
                    perturbed_text += token.text_with_ws
                    display_text += token.text_with_ws

    # --- ERROR HANDLING ---
    if target_word:
        if not target_found:
            raise ValueError(f"The word '{target_word}' is not present in this tweet.")
        if target_found and not changed:
            raise ValueError(f"No {mode} found in WordNet for the word '{target_word}'.")
    else:
        if not changed:
            raise ValueError(f"No word in this tweet has a {mode} in the dictionary.")
    
    return perturbed_text, display_text

def get_single_word_replacement(spacy_token, mode="synonym"):
    """
    Cerca un sinonimo/contrario. Prima prova WordNet, se fallisce (per slang o mancanza di copertura)
    interroga l'API di Datamuse per trovare l'alternativa.
    """
    # 1. TENTATIVO CON WORDNET (Più veloce)
    wn_tag = get_wordnet_pos_from_spacy(spacy_token.pos_)
    replacements = set()
    
    if wn_tag:
        synsets = wordnet.synsets(spacy_token.lemma_, pos=wn_tag)
        for syn in synsets:
            for lemma in syn.lemmas():
                if mode == "synonym" and lemma.name().lower() != spacy_token.lemma_.lower():
                    replacements.add(lemma.name().replace('_', ' '))
                elif mode == "antonym" and lemma.antonyms():
                    replacements.add(lemma.antonyms()[0].name().replace('_', ' '))

    # 2. TENTATIVO CON DATAMUSE (Se WordNet fallisce o non ha trovato nulla)
    if not replacements:
        try:
            word = spacy_token.text.lower()
            if mode == "synonym":
                # ml = meaning like
                url = f"https://api.datamuse.com/words?ml={word}&max=5"
            else:
                # rel_ant = antonyms
                url = f"https://api.datamuse.com/words?rel_ant={word}&max=5"
                
            response = requests.get(url, timeout=2).json()
            for item in response:
                if " " not in item['word']: # Evitiamo frasi, prendiamo solo singole parole
                    replacements.add(item['word'])
        except Exception:
            pass # Se l'API non risponde, ignoriamo e andiamo oltre

    # 3. SELEZIONE FINALE
    if replacements:
        new_word = random.choice(list(replacements))
        # Match original capitalization
        if spacy_token.is_title: 
            new_word = new_word.title()
        elif spacy_token.is_upper: 
            new_word = new_word.upper()
        return new_word
        
    return None

import random

def perturb_noise(text):
    """
    3. Noise Perturbation: Inserts random noise words into the tweet.
    Updated with literature-backed noise categories (Disfluencies, OOV, Slang).
    """
    doc = nlp(text)
    if len(doc) == 0: return text, text
    
    # Extended list based on CheckList (Ribeiro 2020) and Belinkov & Bisk (2018)
    academic_noise_words = [
        # Disfluencies (CheckList)
        "uh", "um", "uhm", "err", "ah", "like", "literally", "basically", "actually", 
        
        # Synthetic / Keyboard mash
        "blah", "xyz", "asdf", "qwerty", "foo", "bar",
        
        # Twitter/Social Media structural noise
        "lol", "lmao", "smh", "tbh", "btw", "ikr", "omg"
    ]
    
    tokens_with_ws = [token.text_with_ws for token in doc]
    display_tokens = [token.text_with_ws for token in doc]
    
    # Insert 1 to 3 random noise words to see cumulative drift
    num_noise = random.randint(1, 3) 
    
    for _ in range(num_noise):
        insert_idx = random.randint(0, len(tokens_with_ws))
        noise = random.choice(academic_noise_words)
        
        # Testo normale per l'embedding
        tokens_with_ws.insert(insert_idx, noise + " ")
        
        # Testo HTML per la UI: evidenziamo in giallo SOLO le parole inserite
        display_tokens.insert(insert_idx, f"<span class='perturbation-diff'>{noise}</span> ")
        
    new_text = "".join(tokens_with_ws)
    display_text = "".join(display_tokens)
    
    return new_text, display_text

def perturb_llm(text, mode="paraphrase"):
    """
    4 & 5. LLM Perturbation: Rephrases or flips sentiment completely.
    """
    if mode == "paraphrase":
        instruction = "Rewrite the following airline review tweet using completely different words, but keep the exact same meaning and sentiment."
    elif mode == "opposite":
        instruction = "Rewrite the following airline review tweet to have the exact opposite sentiment. If it complains, make it praise the airline. If it praises, make it complain."
        
    messages = [
        {
            "role": "system", 
            "content": "You are a helpful assistant specialized in rewriting text. Output ONLY the rewritten text, without explanations, conversational filler, markdown blocks, or quotes."
        },
        {
            "role": "user", 
            "content": f"{instruction}\n\nTweet: {text}"
        }
    ]
    
    text_input = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = gen_tokenizer([text_input], return_tensors="pt").to(gen_model.device)    
    with torch.no_grad():
        generated_ids = gen_model.generate(
            **model_inputs, 
            max_new_tokens=100, # Aumentato per evitare che Gemma tagli le risposte
            temperature=0.7,
            top_p=0.95,         # Raccomandato per Gemma
            top_k=64,           # Raccomandato per Gemma
            do_sample=True
        )

    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    torch.cuda.empty_cache() # Svuota la VRAM dai calcoli temporanei non più necessari
    return gen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


def perturb_typo(text):
    """
    1. Typo Perturbation: Inverte due lettere preservando spazi e formattazione originali.
    """
    doc = nlp(text)
    tokens = [t for t in doc]
    
    candidates = [i for i, t in enumerate(tokens) if len(t.text) > 4 and t.is_alpha]
    
    if not candidates:
        return text, text # Se fallisce, restituisce il testo invariato
        
    target_idx = random.choice(candidates)
    word = tokens[target_idx].text
    
    swap_idx = random.randint(1, len(word) - 3)
    new_word = word[:swap_idx] + word[swap_idx+1] + word[swap_idx] + word[swap_idx+2:]
    
    # Preserva maiuscole
    if tokens[target_idx].is_title: new_word = new_word.title()
    elif tokens[target_idx].is_upper: new_word = new_word.upper()
        
    new_text = ""
    display_text = ""
    
    for i, t in enumerate(tokens):
        if i == target_idx:
            new_text += new_word + t.whitespace_
            # Evidenzia l'errore in arancione sottolineato nella UI!
            display_text += f"<strong style='color: #f97316; text-decoration: underline;'>{new_word}</strong>{t.whitespace_}"
        else:
            new_text += t.text_with_ws
            display_text += t.text_with_ws
            
    return new_text, display_text

def perturb_shuffle(text):
    """
    2. Syntax Attack: Mescola solo le parole vere, ignorando punteggiatura e preservando spazi.
    """
    doc = nlp(text)
    word_tokens = [t for t in doc if t.is_alpha]
    
    if len(word_tokens) <= 2:
        return text, text
        
    shuffled_texts = [t.text for t in word_tokens]
    random.shuffle(shuffled_texts)
    
    new_text = ""
    display_text = ""
    word_idx = 0
    
    for t in doc:
        if t.is_alpha:
            new_word = shuffled_texts[word_idx]
            new_text += new_word + t.whitespace_
            display_text += f"<strong style='color: #8b5cf6;'>{new_word}</strong>{t.whitespace_}"
            word_idx += 1
        else:
            new_text += t.text_with_ws
            display_text += t.text_with_ws
            
    return new_text, display_text

def perturb_leetspeak(text):
    """
    3. Leetspeak Attack: Sostituisce i caratteri con numeri visivamente simili.
    """
    doc = nlp(text)
    leet_map = {'a': '4', 'A': '4', 'e': '3', 'E': '3', 'i': '1', 'I': '1', 'o': '0', 'O': '0', 's': '$', 'S': '$', 't': '7', 'T': '7'}
    
    new_text = ""
    display_text = ""
    changed_any = False
    
    for t in doc:
        if t.is_alpha and len(t.text) > 3:
            perturbed_word = ""
            changed_word = False
            for char in t.text:
                if char in leet_map and random.random() > 0.4:
                    perturbed_word += leet_map[char]
                    changed_word = True
                    changed_any = True
                else:
                    perturbed_word += char
                    
            new_text += perturbed_word + t.whitespace_
            if changed_word:
                display_text += f"<strong style='color: #10b981;'>{perturbed_word}</strong>{t.whitespace_}"
            else:
                display_text += perturbed_word + t.whitespace_
        else:
            new_text += t.text_with_ws
            display_text += t.text_with_ws
            
    if not changed_any: return text, text
    return new_text, display_text



## TF-IDF

In [12]:
def setup_tfidf(testi_globali):
    global global_vectorizer, global_feature_names
    print("Addestramento TF-IDF globale in corso...")
    # Ignoriamo le stopwords (the, is, at) e prendiamo le 2000 parole più importanti
    global_vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
    global_vectorizer.fit(testi_globali)
    global_feature_names = global_vectorizer.get_feature_names_out()
    print("TF-IDF pronto.")

## Bertopic

In [13]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
# Importiamo le stop words inglesi di base
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS 

global_topics_data = []
def initialize_bertopic(tweets_texts, precalculated_embeddings):
    global global_topics_data
    print("Initializing BERTopic (with Custom Stop-Words)...")

    # 1. CREATE THE CUSTOM STOP-WORD LIST
    custom_stop_words = [
        # 1. Nomi compagnie e volo
        "united", "usairways", "americanair", "southwestair", "jetblue", "virginamerica",
        "flight", "flights", "plane", "airline", "airlines", "virgin", "southwest", "american", "fly", "flying",
        
        # 2. Artefatti di Twitter e Stop-words contratte
        "amp", "rt", "http", "https", "co", "im", "dont", "cant", "didnt", "ive", "youre", "ur", "ill",
        
        # 3. Gergo del Customer Care
        "dm", "please", "thanks", "thank", "just", "got", "get",
        "sent", "send", "follow", "following", "help", "appreciate", "let", "know", "need",
        
        # 4. NUOVO: Emozioni, Insulti e Riempitivi colloquiali
        "great", "love", "good", "bad", "fuck", "suck", "sucks", "guys", "okay", "ok", 
        "new", "really", "way", "make", "today", "people", "yes", "right"
    ]

    final_stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words

    # 2. CONFIGURE THE VECTORIZER
    # Reintroduce bigrams to provide more contextual information
    vectorizer_model = CountVectorizer(
        stop_words=final_stop_words,
        ngram_range=(1, 2)
    )

    # 3. INITIALIZE BERTOPIC
    topic_model = BERTopic(
        language="english",
        vectorizer_model=vectorizer_model,
        min_topic_size=15,    # Set back to 15, which previously captured micro-topics effectively
        nr_topics=15,         # No more "AUTO". Force the model to divide the data into 12 topics
        calculate_probabilities=False
    )

    # Compute the topics
    topics, probabilities = topic_model.fit_transform(
        tweets_texts,
        embeddings=precalculated_embeddings
    )

    topic_info = topic_model.get_topic_info()

    for index, row in topic_info.iterrows():
        topic_id = row['Topic']

        # Skip the outlier topic (-1)
        if topic_id == -1:
            continue

        # Retrieve the top 5 most representative words for the topic
        top_words = topic_model.get_topic(topic_id)[:5]

        global_topics_data.append({
            "id": int(topic_id),
            "count": int(row['Count']),
            "name": row['Name'],
            "words": [
                {"word": w[0], "weight": float(w[1])}
                for w in top_words
            ]
        })

    print("BERTopic is ready! Found", len(global_topics_data), "valid topics.")
    return topics

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


## Visual analytics features

In [14]:
global_umap_mapper = None
global_topics_data = []

# 1. Scatterplot using UMAP
def generate_base_map_umap():
    global global_umap_mapper
    global global_topics_data

    # Percorsi dei file. 
    csv_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "base_map.csv")
    model_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR,"umap_mapper.joblib")
    topics_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "topics_bertopic.json")
    embeddings_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "embeddings_1024d.pt")

    tweets_dataset_path = os.path.join(BASE_INPUT_PATH, TWEETS_DIR, "tweets.csv")
    
    # Check if files already exist 
    if os.path.exists(csv_path) and os.path.exists(model_path) and os.path.exists(topics_path):
        print(f"Found existing '{csv_path}' and '{model_path}'.")
        print("Loading UMAP model into memory... (Skipping heavy embedding computation!)")
        global_umap_mapper = joblib.load(model_path)

        df_saved = pd.read_csv(csv_path)
        setup_tfidf(df_saved['text'].astype(str).tolist())
        
        #Carichiamo i topic già calcolati 
        with open(topics_path, 'r') as f:
            global_topics_data = json.load(f)
        print("Loaded BERTopic data from JSON.")
        
        print("System ready.")
        return

    print("Files not found. Loading dataset to generate base map from scratch...")
    
    df_full = pd.read_csv(tweets_dataset_path, dtype={"tweet_id": str})
    df_sample = df_full.copy()

    # TF-IDF
    texts = df_sample['text'].astype(str).tolist()
    setup_tfidf(texts)
    
    texts = df_sample['text'].astype(str).tolist()
    
    # Use the batched function for speed
        
    # If the embeddings were saved but the topics were not,
    # load the embeddings from disk instantly.
    if os.path.exists(embeddings_path):
        print("Saved embeddings found! Loading into RAM...")
        embeddings_1024d = torch.load(embeddings_path)
    else:
        print("No embeddings found. Computing from scratch with the Qwen model (this may take some time)...")
        embeddings_1024d = batch_get_embeddings(texts, batch_size=32)
    
        # Save the computed tensor for future runs
        embeddings_save_path = os.path.join(BASE_WORKING_PATH, "embeddings_1024d.pt")
        torch.save(embeddings_1024d, embeddings_save_path)
        print(f"✅ Embeddings successfully saved to: {embeddings_save_path}")
        # BERTopic
        print("\nExtracting Topics with BERTopic...")
    
    document_topics = initialize_bertopic(texts, embeddings_1024d.numpy()) 
    
    # Salviamo i risultati nel JSON
    topics_save_path = os.path.join(BASE_WORKING_PATH, "topics_bertopic.json")
    with open(topics_save_path, 'w') as f:
        json.dump(global_topics_data, f)
    print("Topics saved successfully to JSON.")
    
    print("\nTraining UMAP for 2D projection...")
    
    # Inizializza la UMAP per 2 componenti
    umap_model = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        metric='cosine', 
        random_state=42
    )
    
    # Fit e transform per ottenere le coordinate 2D
    embeddings_2d = umap_model.fit_transform(embeddings_1024d.numpy())
    
    # Salviamo l'oggetto UMAP nella variabile globale
    global_umap_mapper = umap_model
    
    df_map = pd.DataFrame({
        "tweet_id": df_sample["tweet_id"],
        "text": df_sample["text"],
        "sentiment": df_sample["airline_sentiment"],
        "airline": df_sample["airline"],
        "x": embeddings_2d[:, 0], # x coordinates
        "y": embeddings_2d[:, 1],  # y coordinates
        "topic_id": document_topics # topic of the tweet
    })
    csv_save_path = os.path.join(BASE_WORKING_PATH, "base_map.csv")
    model_save_path = os.path.join(BASE_WORKING_PATH, "umap_mapper.joblib")
    
    df_map.to_csv(csv_save_path, index=False)
    print("Base map generated successfully and saved as 'base_map.csv'.")
    
    # Salva l'oggetto UMAP
    joblib.dump(global_umap_mapper, model_save_path)

# 2. Token weight analysis: take one tweet, and removes 1 token at a time, recomputes the embeddings of the new tweet
# without that token to measure the cosine similarity between the original embedding and the new one


embedding_lock = threading.Lock()

def get_token_weights(tweet_text):
    with embedding_lock:
        orig_emb = get_embedding(tweet_text)

        # Using SPACY to divide the tweet in words
        doc = nlp(tweet_text)
        words = [token.text_with_ws for token in doc]

        results = []
        for i, word in enumerate(words):
            if not word.strip(): continue # Skip empty space
                
            # Remove the word at position i 
            mod_words = words.copy()
            mod_words.pop(i)
            mod_text = "".join(mod_words)
            
            if not mod_text.strip(): mod_text = "empty"
                
            # Compute the impact of removing that word
            mod_emb = get_embedding(mod_text)
            sim = F.cosine_similarity(orig_emb, mod_emb).item()
            impact = 1.0 - sim
            
            results.append({
                "word": word.strip(),
                "weight": round(impact, 4)
            })
        return results


# 3. 
def generate_heatmap_data(tweet_text):
    with embedding_lock:
        """Generates the flattened matrix data for the D3 Heatmap."""
        orig_emb = get_embedding(tweet_text)
        doc = nlp(tweet_text)
        
        # Filter only words for the heatmap columns
        valid_tokens = [t for t in doc if t.is_alpha]
        tokens_subset = valid_tokens[:10] # Take the first 10 words
        
        perturbations = ["Remove", "Dict_Synonym", "Dict_Antonym"]
        heatmap_data = []
        
        for token in tokens_subset:
            for pert in perturbations:
                # Default: no modification
                mod_text = tweet_text 
                importance = 0.0
                
                if pert == "Remove":
                    # Reconstruct the string omitting exactly this token's text
                    mod_text = "".join([t.text_with_ws if t.i != token.i else t.whitespace_ for t in doc])
                    
                elif pert == "Dict_Synonym":
                    replacement = get_single_word_replacement(token, mode="synonym")
                    if replacement:
                        # Reconstruct the string substituting only this exact token
                        mod_text = "".join([t.text_with_ws if t.i != token.i else replacement + t.whitespace_ for t in doc])
                        
                elif pert == "Dict_Antonym":
                    replacement = get_single_word_replacement(token, mode="antonym")
                    if replacement:
                        mod_text = "".join([t.text_with_ws if t.i != token.i else replacement + t.whitespace_ for t in doc])
                
                # Calculate new embedding ONLY if the text was actually modified.
                # If no synonym/antonym was found, mod_text == tweet_text, so importance remains 0.0
                if mod_text != tweet_text:
                    if not mod_text.strip(): 
                        mod_text = "empty"
                        
                    mod_emb = get_embedding(mod_text)
                    sim = F.cosine_similarity(orig_emb, mod_emb).item()
                    importance = 1.0 - sim
                
                heatmap_data.append({
                    "token": token.text,
                    "perturbation": pert,
                    "importance": round(importance, 4)
                })
                
        return heatmap_data

def get_perturbed_coordinates(tweet_text, perturbation_type="llm_paraphrase", target_word=None, replacement_word=None):
    """
    Applies one of the 5 specific perturbations, embeds, and projects to UMAP.
    Valid perturbation_types: 'dict_synonym', 'dict_antonym', 'noise', 'llm_paraphrase', 'llm_opposite'
    """
    # Controllo di sicurezza per la UMAP
    if global_umap_mapper is None:
        raise ValueError("UMAP mapper not initialized. Run generate_base_map_umap() first.")

    display_text = None # Default
        
    # Route to the correct perturbation function
    if perturbation_type == "dict_synonym":
        # DEVE ESSERE COSÌ: new_text, display_text = ...
        new_text, display_text = perturb_dict(tweet_text, mode="synonym", target_word=target_word)
    elif perturbation_type == "dict_antonym":
        # DEVE ESSERE COSÌ: new_text, display_text = ...
        new_text, display_text = perturb_dict(tweet_text, mode="antonym", target_word=target_word)
    elif perturbation_type == "dict_remove":
        new_text, display_text = perturb_dict(tweet_text, mode="remove", target_word=target_word)
    elif perturbation_type == "custom":
        new_text, display_text = perturb_dict(tweet_text, mode="custom", target_word=target_word, replacement_word=replacement_word)
    
    elif perturbation_type == "noise":
        new_text, display_text = perturb_noise(tweet_text)
    elif perturbation_type == "llm_paraphrase":
        new_text = perturb_llm(tweet_text, mode="paraphrase")
    elif perturbation_type == "llm_opposite":
        new_text = perturb_llm(tweet_text, mode="opposite")
    elif perturbation_type == "typo":
        new_text, display_text = perturb_typo(tweet_text) # <-- Aggiunto , display_text
    elif perturbation_type == "shuffle":
        new_text, display_text = perturb_shuffle(tweet_text) # <-- Aggiunto , display_text
    elif perturbation_type == "leetspeak":
        new_text, display_text = perturb_leetspeak(tweet_text) # <-- Aggiunto , display_text
    else:
        new_text = tweet_text
        
    if display_text is None:
        display_text = new_text
    new_emb_tensor = get_embedding(new_text)
    new_emb_numpy = new_emb_tensor.cpu().numpy()
    
    # Project into existing 2D space
    new_2d_coords = global_umap_mapper.transform(new_emb_numpy)
    
    return {
        "original_text": tweet_text,
        "new_text": new_text,
        "display_text": display_text,
        "perturbation_applied": perturbation_type,
        "new_x": float(new_2d_coords[0][0]),
        "new_y": float(new_2d_coords[0][1])
    }

## Main logic

In [ ]:
import warnings
import os
import subprocess
warnings.filterwarnings('ignore')

if __name__ == "__main__":
    # 1. Generate the Base Map and train the UMAP model in memory
    print("--- STEP 1: INITIALIZING SYSTEM ---")
    generate_base_map_umap()
    
    # 2. Set up the Flask API Server
    print("\n--- STEP 2: STARTING API SERVER ---")
    
    # We install the OFFICIAL ngrok library (which bypasses the 403 error entirely)
    print("Installing required server packages...")
    subprocess.run(["pip", "install", "-q", "ngrok", "flask", "flask-cors"])
    
    from flask import Flask, request, jsonify
    from flask_cors import CORS
    import ngrok
    from kaggle_secrets import UserSecretsClient

    app = Flask(__name__)
    CORS(app)

    @app.route('/api/perturb', methods=['POST'])
    def perturb_endpoint():
        data = request.json
        tweet_text = data.get('text')
        perturb_type = data.get('type', 'llm_paraphrase') 
        target_word = data.get('target_word', None) 
        replacement_word = data.get('replacement_word', None)
        
        try:
            with embedding_lock:
                result = get_perturbed_coordinates(
                    tweet_text, 
                    perturb_type, 
                    target_word=target_word, 
                    replacement_word=replacement_word
                )
            return jsonify(result)
        except Exception as e:
            # Se Python solleva il ValueError, lo rimanda al frontend
            return jsonify({"error": str(e)}), 500

    @app.route('/api/tfidf', methods=['POST'])
    def tfidf_endpoint():
        data = request.json
        selected_texts = data.get('texts', [])
        
        if not selected_texts or global_vectorizer is None:
            return jsonify([])
    
        # 1. Calcola il TF-IDF solo per i testi selezionati, basandosi sul vocabolario globale
        subset_matrix = global_vectorizer.transform(selected_texts)
        
        # 2. Somma i punteggi di ogni parola per vedere quali dominano in questo cluster
        sum_scores = subset_matrix.sum(axis=0)
        scores_array = np.squeeze(np.asarray(sum_scores))
        
        # 3. Trova gli indici delle 10 parole con il punteggio più alto
        top_indices = scores_array.argsort()[-10:][::-1]
        
        # 4. Formatta il risultato in JSON per D3.js
        results = []
        for i in top_indices:
            if scores_array[i] > 0: # Ignora parole con score 0
                results.append({
                    "word": global_feature_names[i],
                    "score": float(scores_array[i])
                })
                
        return jsonify(results)

    @app.route('/api/token_weights', methods=['POST'])
    def token_weights_endpoint():
        data = request.json
        tweet_text = data.get('text')
        try:
            result = get_token_weights(tweet_text)
            return jsonify(result)
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/heatmap', methods=['POST'])
    def heatmap_endpoint():
        data = request.json
        tweet_text = data.get('text')
        try:
            result = generate_heatmap_data(tweet_text)
            return jsonify(result)
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/topics', methods=['GET'])
    def get_topics():
        # Restituisce i topic calcolati all'avvio
        return jsonify(global_topics_data)

    # --- KAGGLE NGROK SETUP (OFFICIAL SDK METHOD) ---
    print("Configuring ngrok...")
    try:
        user_secrets = UserSecretsClient()
        my_secret_token = user_secrets.get_secret("NGROK_TOKEN")
        
        listener = await ngrok.forward(PORT, authtoken=my_secret_token)
        public_url = listener.url()
        
        print(f"\n✅ YOUR API IS LIVE AT: {public_url}")
        print("Copy this URL into your app.js fetch() calls!")
        
        # Avviamo il server nel blocco try
        print("\n⏳ Server in ascolto... Premi il tasto 'Stop' (⏹️) accanto alla cella per spegnerlo in modo sicuro.")
        app.run(port=PORT, threaded=False)

    except KeyboardInterrupt:
        # QUESTO SCATTA QUANDO PREMI IL TASTO STOP SU KAGGLE
        print("\n🛑 Interruzione rilevata (Ctrl+C). Chiusura del tunnel in corso...")
        listener.close() # Chiude il tunnel Ngrok liberando l'account!
        print("✅ Tunnel Ngrok chiuso e server spento in modo pulito.")
        
    except Exception as e:
        print("\n⚠️ Errore imprevisto:")
        raise e

--- STEP 1: INITIALIZING SYSTEM ---
Found existing '/kaggle/input/datasets/chiaracervelli/embedding/base_map.csv' and '/kaggle/input/datasets/chiaracervelli/embedding/umap_mapper.joblib'.
Loading UMAP model into memory... (Skipping heavy embedding computation!)
Addestramento TF-IDF globale in corso...
TF-IDF pronto.
Loaded BERTopic data from JSON.
System ready.

--- STEP 2: STARTING API SERVER ---
Installing required server packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 42.1 MB/s eta 0:00:00
Configuring ngrok...

✅ YOUR API IS LIVE AT: https://strike-happily-lurk.ngrok-free.dev
Copy this URL into your app.js fetch() calls!

⏳ Server in ascolto... Premi il tasto 'Stop' (⏹️) accanto alla cella per spegnerlo in modo sicuro.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5500
Press CTRL+C to quit
127.0.0.1 - - [16/May/2026 13:00:41] "OPTIONS /api/topics HTTP/1.1" 200 -
127.0.0.1 - - [16/May/2026 13:00:42] "GET /api/topics HTTP/1.1" 200 -
